# Question 1 — Brick-wall inspection

**Input:** `brick.png` (8-bit grayscale, 256 × 256).

This notebook implements both requested styles:
1. OpenCV built-in image-processing functions.
2. `cv2.imread()` only for reading; NumPy implementations for all processing, with Matplotlib used only for display.

The second implementation deliberately does not call OpenCV processing functions. Run the setup cell, set the input filename if needed, then run either or both sections.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import heapq
from collections import Counter

IMAGE_PATH = "brick.png"  # Put brick.png beside this notebook, or change this path.

def show_gray(title, image, cmap="gray"):
    plt.figure(figsize=(7, 5))
    plt.imshow(image, cmap=cmap, vmin=0, vmax=255 if image.dtype != bool else 1)
    plt.title(title)
    plt.axis("off")
    plt.show()

def load_gray_cv2(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Could not read {path}. Place brick.png beside this notebook.")
    if image.shape != (256, 256):
        print(f"Input shape is {image.shape}; the question specifies 256x256.")
    return image

## Style 1 — OpenCV built-in methods

Pipeline: 5 × 5 Gaussian smoothing (σ = 1, replicated border) → Sobel gradient magnitude → threshold at 80 → 3 × 3 morphological closing → 4-connected components. Components touching the image border or having fewer than 3 pixels are excluded. The centroid is calculated from component pixel coordinates.

In [ ]:
img1 = load_gray_cv2(IMAGE_PATH)

# Gaussian smoothing with replicated border.
smooth1 = cv2.GaussianBlur(img1, (5, 5), sigmaX=1, sigmaY=1,
                           borderType=cv2.BORDER_REPLICATE)

# Sobel gradients (3x3); gradient magnitude.
gx1 = cv2.Sobel(smooth1, cv2.CV_32F, 1, 0, ksize=3)
gy1 = cv2.Sobel(smooth1, cv2.CV_32F, 0, 1, ksize=3)
mag1 = np.sqrt(gx1**2 + gy1**2)
edge1 = (mag1 >= 80).astype(np.uint8) * 255

kernel1 = np.ones((3, 3), np.uint8)
closed1 = cv2.morphologyEx(edge1, cv2.MORPH_CLOSE, kernel1,
                           borderType=cv2.BORDER_CONSTANT,
                           borderValue=0)

# 4-connected components.
n1, labels1, stats1, cents1 = cv2.connectedComponentsWithStats(
    (closed1 == 0).astype(np.uint8), connectivity=4
)
regions1 = []
for lab in range(1, n1):
    x, y, w, h, area = stats1[lab]
    cx, cy = cents1[lab]
    touches = (x == 0 or y == 0 or x + w == img1.shape[1] or y + h == img1.shape[0])
    if not touches and area >= 9 and w >= 3 and h >= 3:
        regions1.append((lab, int(area), float(cx), float(cy)))

# Display labels over the original image.
annotated1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
for idx, (lab, area, cx, cy) in enumerate(regions1, start=1):
    cv2.circle(annotated1, (round(cx), round(cy)), 3, (255, 0, 0), -1)
    cv2.putText(annotated1, str(idx), (round(cx)+3, round(cy)-3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1)

print("Detected bricks:", len(regions1))
print("Regions (label, area, centroid_x, centroid_y):")
for row in regions1:
    print(row)

show_gray("Original grayscale image", img1)
show_gray("Gaussian smoothed", smooth1)
show_gray("Binary Sobel edge map", edge1)
show_gray("Morphological closing", closed1)
plt.figure(figsize=(8, 6)); plt.imshow(annotated1); plt.title("Numbered brick regions"); plt.axis("off"); plt.show()

### Huffman compression and final survey image (Style 1)

The grayscale image is Huffman encoded using its pixel-value frequencies. The notebook reports entropy, average code length, coding efficiency, compressed size, compression ratio and exact reconstruction. The final RGB survey image highlights detected brick pixels in blue and leaves other pixels unchanged.

In [ ]:
def build_huffman_codes(freq):
    heap = []
    serial = 0
    for symbol, count in freq.items():
        heapq.heappush(heap, (count, serial, {"symbol": symbol}))
        serial += 1
    if len(heap) == 1:
        return {heap[0][2]["symbol"]: "0"}
    while len(heap) > 1:
        f1, _, n1 = heapq.heappop(heap)
        f2, _, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, serial, {"left": n1, "right": n2}))
        serial += 1
    codes = {}
    def walk(node, prefix):
        if "symbol" in node:
            codes[node["symbol"]] = prefix or "0"
        else:
            walk(node["left"], prefix + "0")
            walk(node["right"], prefix + "1")
    walk(heap[0][2], "")
    return codes

def huffman_report(gray):
    vals, counts = np.unique(gray, return_counts=True)
    freq = dict(zip(vals.tolist(), counts.tolist()))
    total = gray.size
    p = counts / total
    entropy = float(-np.sum(p * np.log2(p)))
    codes = build_huffman_codes(freq)
    avg_len = sum(freq[s] * len(codes[s]) for s in freq) / total
    compressed_bits = sum(freq[s] * len(codes[s]) for s in freq)
    original_bits = total * 8
    efficiency = entropy / avg_len * 100 if avg_len else 0
    ratio = original_bits / compressed_bits if compressed_bits else float("inf")
    # Encode and decode the complete sequence to verify exact reconstruction.
    bitstream = "".join(codes[int(v)] for v in gray.ravel())
    reverse = {v: k for k, v in codes.items()}
    decoded, current = [], ""
    for bit in bitstream:
        current += bit
        if current in reverse:
            decoded.append(reverse[current]); current = ""
    reconstructed = np.array(decoded, dtype=np.uint8).reshape(gray.shape)
    print(f"Entropy H                 : {entropy:.4f} bits/pixel")
    print(f"Average code length       : {avg_len:.4f} bits/pixel")
    print(f"Coding efficiency         : {efficiency:.2f}%")
    print(f"Compressed size           : {compressed_bits} bits")
    print(f"Compression ratio         : {ratio:.4f}")
    print(f"Reconstruction exact      : {np.array_equal(gray, reconstructed)}")
    return reconstructed, codes, bitstream

reconstructed1, codes1, bitstream1 = huffman_report(img1)

# Highlight pixels belonging to retained components in blue.
brick_mask1 = np.isin(labels1, [r[0] for r in regions1])
final1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
final1[brick_mask1] = [0, 0, 255]
plt.figure(figsize=(8, 6)); plt.imshow(final1); plt.title(f"Final survey image — {len(regions1)} bricks (blue)"); plt.axis("off"); plt.show()

## Style 2 — NumPy implementation (OpenCV only for `imread`)

All filtering, Sobel gradients, thresholding, closing, connected-component labeling, centroid calculation, Huffman coding and final visualization are implemented manually. `cv2` is used only in the image-reading line.

In [ ]:
# cv2.imread is the ONLY OpenCV operation in this section.
img2 = cv2.imread(IMAGE_PATH, 0)
if img2 is None:
    raise FileNotFoundError(f"Could not read {IMAGE_PATH}")
img2 = img2.astype(np.uint8)

def correlate2d_replicate(a, kernel):
    a = a.astype(float); kh, kw = kernel.shape
    py, px = kh//2, kw//2
    padded = np.pad(a, ((py, py), (px, px)), mode="edge")
    out = np.zeros_like(a, dtype=float)
    for i in range(kh):
        for j in range(kw):
            out += kernel[i, j] * padded[i:i+a.shape[0], j:j+a.shape[1]]
    return out

def gaussian_kernel(size=5, sigma=1.0):
    ax = np.arange(-(size//2), size//2+1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx**2 + yy**2)/(2*sigma**2))
    return k / k.sum()

smooth2 = correlate2d_replicate(img2, gaussian_kernel(5, 1.0))
sobel_x = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float)
sobel_y = sobel_x.T
gx2 = correlate2d_replicate(smooth2, sobel_x)
gy2 = correlate2d_replicate(smooth2, sobel_y)
mag2 = np.sqrt(gx2**2 + gy2**2)
edge2 = mag2 >= 80

def binary_dilate(mask, kernel):
    h, w = mask.shape; kh, kw = kernel.shape
    py, px = kh//2, kw//2
    p = np.pad(mask, ((py,py),(px,px)), mode="constant", constant_values=False)
    out = np.zeros((h,w), bool)
    for i in range(kh):
        for j in range(kw):
            if kernel[i,j]:
                out |= p[i:i+h, j:j+w]
    return out

def binary_erode(mask, kernel):
    h, w = mask.shape; kh, kw = kernel.shape
    py, px = kh//2, kw//2
    p = np.pad(mask, ((py,py),(px,px)), mode="constant", constant_values=False)
    out = np.ones((h,w), bool)
    for i in range(kh):
        for j in range(kw):
            if kernel[i,j]:
                out &= p[i:i+h, j:j+w]
    return out

k3 = np.ones((3,3), bool)
closed2 = binary_erode(binary_dilate(edge2, k3), k3)

In [ ]:
def connected_components_manual(mask, connectivity=4):
    h, w = mask.shape
    labels = np.zeros((h,w), dtype=np.int32)
    if connectivity == 4:
        offsets = [(-1,0),(1,0),(0,-1),(0,1)]
    else:
        offsets = [(dy,dx) for dy in (-1,0,1) for dx in (-1,0,1) if (dy,dx)!=(0,0)]
    components = []
    lab = 0
    for y in range(h):
        for x in range(w):
            if mask[y,x] and labels[y,x] == 0:
                lab += 1; stack = [(y,x)]; labels[y,x] = lab; pixels=[]
                while stack:
                    cy,cx = stack.pop(); pixels.append((cy,cx))
                    for dy,dx in offsets:
                        ny,nx = cy+dy,cx+dx
                        if 0 <= ny < h and 0 <= nx < w and mask[ny,nx] and labels[ny,nx] == 0:
                            labels[ny,nx] = lab; stack.append((ny,nx))
                ys = np.array([p[0] for p in pixels]); xs = np.array([p[1] for p in pixels])
                components.append({"label":lab,"area":len(pixels),"cx":xs.mean(),"cy":ys.mean(),
                                   "x":xs.min(),"y":ys.min(),"x2":xs.max(),"y2":ys.max()})
    return labels, components

labels2, comps2 = connected_components_manual(~closed2, 4)
regions2 = []
for c in comps2:
    touches = c["x"] == 0 or c["y"] == 0 or c["x2"] == img2.shape[1]-1 or c["y2"] == img2.shape[0]-1
    if not touches and c["area"] >= 9 and (c["x2"]-c["x"]+1) >= 3 and (c["y2"]-c["y"]+1) >= 3:
        regions2.append(c)

print("Detected bricks:", len(regions2))
for c in regions2:
    print(c["label"], c["area"], round(c["cx"],2), round(c["cy"],2))

def build_huffman_codes_numpy(freq):
    heap = []
    serial = 0
    for symbol, count in freq.items():
        heapq.heappush(heap, (count, serial, {"symbol": symbol}))
        serial += 1
    if len(heap) == 1:
        return {heap[0][2]["symbol"]: "0"}
    while len(heap) > 1:
        f1, _, n1 = heapq.heappop(heap)
        f2, _, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, serial, {"left": n1, "right": n2}))
        serial += 1
    codes = {}
    def walk(node, prefix):
        if "symbol" in node:
            codes[node["symbol"]] = prefix or "0"
        else:
            walk(node["left"], prefix + "0")
            walk(node["right"], prefix + "1")
    walk(heap[0][2], "")
    return codes

def huffman_numpy_report(gray):
    vals, counts = np.unique(gray, return_counts=True)
    freq = dict(zip(vals.tolist(), counts.tolist()))
    total = gray.size; probs = counts / total
    entropy = float(-np.sum(probs * np.log2(probs)))
    codes = build_huffman_codes_numpy(freq)
    avg = sum(freq[s]*len(codes[s]) for s in freq)/total
    bits = sum(freq[s]*len(codes[s]) for s in freq)
    stream = "".join(codes[int(v)] for v in gray.ravel())
    rev = {v:k for k,v in codes.items()}
    decoded=[]; cur=""
    for bit in stream:
        cur += bit
        if cur in rev: decoded.append(rev[cur]); cur=""
    recon=np.asarray(decoded,dtype=np.uint8).reshape(gray.shape)
    print(f"Entropy: {entropy:.4f} bits/pixel")
    print(f"Average code length: {avg:.4f} bits/pixel")
    print(f"Coding efficiency: {entropy/avg*100:.2f}%")
    print(f"Compressed size: {bits} bits")
    print(f"Compression ratio: {gray.size*8/bits:.4f}")
    print("Reconstruction exact:", np.array_equal(gray,recon))
    return recon

reconstructed2 = huffman_numpy_report(img2)

mask2 = np.isin(labels2, [c["label"] for c in regions2])
final2 = np.repeat(img2[:,:,None], 3, axis=2)
final2[mask2] = [0,0,255]  # RGB blue
show_gray("Original", img2)
show_gray("Gaussian smoothed", smooth2)
show_gray("Binary Sobel edge map", edge2.astype(np.uint8)*255)
show_gray("Morphological closing", closed2.astype(np.uint8)*255)
plt.figure(figsize=(8,6)); plt.imshow(final2); plt.title(f"Final survey image — {len(regions2)} bricks (blue)"); plt.axis("off"); plt.show()

### Notes
- The problem's sample output is illustrative; exact brick counts depend on the supplied `brick.png` and image quality.
- Huffman compressed size above counts encoded payload bits only; it excludes the codebook/header and byte-alignment overhead.
- Connected-component filtering follows the stated rule: remove border-touching components and components with area below 3 pixels.